# Aula 08 - Notebook: Sistemas Especialistas — Base de Conhecimento e Regras de Diagnóstico

Neste notebook implementamos a arquitetura de **Base de Conhecimento Industrial** orientada a objetos para o комплекс de fertilizantes. Estruturamos fatos, regras de produção em Cláusulas de Horn, mecanismos de verificação de consistência e exportação de relatórios estruturados.


In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

from dataclasses import dataclass, field
from typing import List, Set, Dict, Any, Optional
import time

@dataclass
class Fato:
    nome: str
    valor: bool
    descricao: str
    fonte: str = "SENSOR" # 'SENSOR' ou 'INFERIDO'
    timestamp: float = field(default_factory=time.time)

@dataclass
class RegraDiagnostico:
    id_regra: str
    antecedentes: Set[str]
    consequente: str
    descricao_diagnostico: str
    severidade: str      # 'CRÍTICA', 'ALTA', 'MÉDIA', 'BAIXA'
    prioridade: int      # 1 a 10 (10 = mais urgente)
    tempo_resposta_max_s: float
    procedimento_pop: str

class BaseConhecimentoSCADA:
    def __init__(self):
        self.regras: List[RegraDiagnostico] = []
        self._indice_antecedentes: Dict[str, List[RegraDiagnostico]] = {}

    def adicionar_regra(
        self, id_regra: str, antecedentes: List[str], consequente: str,
        descricao: str, severidade: str = "ALTA", prioridade: int = 5,
        tempo_max_s: float = 5.0, pop: str = "Verificar malha"
    ):
        regra = RegraDiagnostico(
            id_regra=id_regra,
            antecedentes=set(antecedentes),
            consequente=consequente,
            descricao_diagnostico=descricao,
            severidade=severidade,
            prioridade=prioridade,
            tempo_resposta_max_s=tempo_max_s,
            procedimento_pop=pop
        )
        self.regras.append(regra)
        
        for ant in antecedentes:
            if ant not in self._indice_antecedentes:
                self._indice_antecedentes[ant] = []
            self._indice_antecedentes[ant].append(regra)

    def obter_regras_por_fato(self, fato_nome: str) -> List[RegraDiagnostico]:
        return self._indice_antecedentes.get(fato_nome, [])

    def exportar_catalogo(self) -> List[Dict[str, Any]]:
        catalogo = []
        for r in sorted(self.regras, key=lambda x: x.prioridade, reverse=True):
            catalogo.append({
                "ID": r.id_regra,
                "Prioridade": r.prioridade,
                "Severidade": r.severidade,
                "SE (Antecedentes)": " AND ".join(sorted(r.antecedentes)),
                "ENTÃO (Consequente)": r.consequente,
                "Diagnóstico": r.descricao_diagnostico,
                "POP": r.procedimento_pop
            })
        return catalogo

bc = BaseConhecimentoSCADA()

# Cadastro das regras da planta de fertilizantes
bc.adicionar_regra(
    "R-01", ["p1", "t1"], "REACAO_RUNAWAY",
    "Exotermia Descontrolada no Reator R-101", "CRÍTICA", 10, 1.0,
    "POP-SIS-01: Desarme total imediato e abertura de resfriamento"
)
bc.adicionar_regra(
    "R-02", ["REACAO_RUNAWAY", "v1"], "TRIP_ALIMENTACAO_NH3",
    "Corte Imediato de Alimentação de Amônia", "CRÍTICA", 10, 0.5,
    "POP-SIS-02: Fechar XV-101 e despressurizar linha para flare"
)
bc.adicionar_regra(
    "R-03", ["l_low", "m1"], "CAVITACAO_BOMBA_P101",
    "Risco de Cavitação e Destruição de P-101", "ALTA", 8, 2.0,
    "POP-MA-04: Desligar bomba P-101 e verificar nível de ácido"
)
bc.adicionar_regra(
    "R-04", ["g1"], "FUGA_TOXICA_NH3",
    "Vazamento Atmosférico de Amônia no Setor 100", "CRÍTICA", 9, 1.0,
    "POP-SST-08: Acionar sirene de evacuação e cortina de água"
)
bc.adicionar_regra(
    "R-05", ["f1", "c1"], "FALHA_COMBUSTAO_SECADOR",
    "Extinção de Chama no Queimador do Secador 201", "ALTA", 7, 3.0,
    "POP-SEC-02: Fechar XV-201, cortar gás e purgar câmara"
)
bc.adicionar_regra(
    "R-06", ["p_crio", "t_crio"], "SOBREPRESSAO_TANQUE_CRIO",
    "Boil-Off Excessivo no Tanque Criogênico TK-301", "CRÍTICA", 9, 2.0,
    "POP-CRIO-01: Acionar compressores de recuperação de gás"
)

print("=== CATÁLOGO OFICIAL DA BASE DE CONHECIMENTO DO SCADA-CORE ===")
print(formatar_tabela(bc.exportar_catalogo()))

assert len(bc.regras) == 6
assert len(bc.obter_regras_por_fato("p1")) >= 1
print("\n[OK] Base de Conhecimento estruturada, indexada e validada com sucesso!")


=== CATÁLOGO OFICIAL DA BASE DE CONHECIMENTO DO SCADA-CORE ===
ID   | Prioridade | Severidade | SE (Antecedentes)     | ENTÃO (Consequente)      | Diagnóstico                                    | POP                                                          
-----+------------+------------+-----------------------+--------------------------+------------------------------------------------+--------------------------------------------------------------
R-01 | 10         | CRÍTICA    | p1 AND t1             | REACAO_RUNAWAY           | Exotermia Descontrolada no Reator R-101        | POP-SIS-01: Desarme total imediato e abertura de resfriamento
R-02 | 10         | CRÍTICA    | REACAO_RUNAWAY AND v1 | TRIP_ALIMENTACAO_NH3     | Corte Imediato de Alimentação de Amônia        | POP-SIS-02: Fechar XV-101 e despressurizar linha para flare  
R-04 | 9          | CRÍTICA    | g1                    | FUGA_TOXICA_NH3          | Vazamento Atmosférico de Amônia no Setor 100   | POP-SST-08: Acionar sire